# Stage 1 — Obtain the data

This notebook downloads every series used in the project from the U.S. Energy Information Administration (EIA), converts each one to a CSV file named after the metric it measures, and writes `data/raw/manifest.csv`, which records the coverage of each file.

| Folder | Content | In Git? |
|---|---|---|
| `data/raw/xls/` | original EIA downloads (`.xls`) | no |
| `data/raw/weekly/` | weekly petroleum data as CSV, e.g. `crude_stocks.csv` (public domain) | yes |
| `data/raw/prices/` | daily WTI and Brent prices as CSV (third-party data, see README) | no |

Files that are already present are not downloaded again, so the notebook can be re-run quickly.

*Written with the help of Claude (Anthropic), which was asked to produce the data-download notebook; declared in the paper appendix.*

In [ ]:
import sys
from pathlib import Path

# Find the repository root (the folder containing src/) so `src` can be imported
ROOT = Path.cwd().resolve()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import MANIFEST_PATH
from src.data_utils import build_manifest, convert_xls_to_csv, download_all, read_series, series_table

pd.set_option("display.max_colwidth", 80)
print("Repository root:", ROOT)

## 1. The series used

In [ ]:
series_table()

## 2. Download the original EIA files

Each file is about 100–500 KB. If a download fails, the reason is shown in the `status` column and the other files are still downloaded.

In [ ]:
status = download_all()
status

## 3. Convert to one CSV per metric

Each `.xls` file is recognised from the EIA code stored inside it and saved as `date, <metric>`.

In [ ]:
converted = convert_xls_to_csv()
converted

## 4. Check the files and write the manifest

In [ ]:
manifest = build_manifest()
manifest[["metric", "frequency", "status", "first_date", "last_date", "n_obs", "n_missing"]]

In [ ]:
problems = manifest[manifest["status"] != "ok"]
if not problems.empty:
    raise RuntimeError(
        "These series are missing or unreadable:\n"
        + problems[["metric", "status", "url"]].to_string(index=False)
        + "\nDownload each file from its url in a browser, copy it to data/raw/xls/, "
        "then run this notebook again."
    )
print(f"All {len(manifest)} metrics are available as CSV.")
print("Manifest saved to", MANIFEST_PATH.relative_to(ROOT))

## 5. Sanity check

The WTI spot price was negative on 20 April 2020. Seeing that value here confirms the price file was converted correctly; it is dealt with in stage 2.

In [ ]:
wti = read_series("wti")
wti.loc["2020-04-17":"2020-04-22"]

**Next:** `02_clean_data.ipynb` turns these files into one weekly table aligned to EIA release dates.